In [1]:
# pip install sqlalchemy psycopg2-binary

# Loading sql tables to pandas dataframes

In this part, we are going to load the tables already created by using the sql files in the `sql`folder of this project. To do that, we are using `sqlalchemy` library in python in order to read SQL queries and `extract.py` file where the connection of the database is done. Remember to modify the credentials in the `config.yaml` to setup your login access.

In [2]:
import pandas as pd
from sqlalchemy import create_engine
import extract

engine = extract.get_engine()

raw_data = extract.extract_table("loan_approval_prediction_raw")
customers = extract.extract_table("staging_customers")
financials = extract.extract_table("staging_financials")
loans = extract.extract_table("staging_loans")

# engine.close()

# Exploring the datasets and making the feature engineering

## Customers dataset

To beging with, let's print a general information of the `customers` dataframe (the firts rows of the dataframe) together with the info() command to detect null information.

In [3]:
customers.head()

,customer_id,loan_id,gender,married,education,dependents
0,1,LP001002,Male,No,Graduate,0.0
1,2,LP001003,Male,Yes,Graduate,1.0
2,3,LP001005,Male,Yes,Graduate,0.0
3,4,LP001006,Male,Yes,Not Graduate,0.0
4,5,LP001008,Male,No,Graduate,0.0


In [4]:
customers.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  598 non-null    int64  
 1   loan_id      598 non-null    object 
 2   gender       598 non-null    object 
 3   married      598 non-null    object 
 4   education    598 non-null    object 
 5   dependents   586 non-null    float64
dtypes: float64(1), int64(1), object(4)
memory usage: 28.2+ KB


As it can be seen from above, the `customers` dataframe contains 6 columns in total, which includes the following information:

* `customer_id`: It is an identification number for each customer.
* `loan_id`: It is an identification serial for each loan of each customer.
* `gender`: If the customer is male or female.
* `married`: If the customer is married or not.
* `education`: If the customer has an graduation degree or not.
* `dependents`: How many dependents each customer has.

The dataset has 598 rows, from which only the *dependents* row contains null values. Let's explore which customers do not have dependents information:

In [5]:
# Find the list of customers with no information regarding their dependents 
customers[customers["dependents"].isnull()]

,customer_id,loan_id,gender,married,education,dependents
101,102,LP001350,Male,Yes,Graduate,NaN
118,119,LP001426,Male,Yes,Graduate,NaN
221,222,LP001754,Male,Yes,Not Graduate,NaN
287,288,LP001945,Female,No,Graduate,NaN
295,296,LP001972,Male,Yes,Not Graduate,NaN
325,326,LP002100,Male,No,Graduate,NaN
327,328,LP002106,Male,Yes,Graduate,NaN
338,339,LP002130,Male,Yes,Not Graduate,NaN
347,348,LP002144,Female,No,Graduate,NaN
504,505,LP002682,Male,Yes,Not Graduate,NaN


Let's also find if the `customers` dataframe contains repeated customers or loan ids. To do that, we can use the method `duplicated()` that allows us to check for all the values in a group of columns if some of them are repeated:

In [6]:
# Finding rows with the same customer_id value
customers[customers.duplicated(subset=['customer_id'], keep=False) == True]

,customer_id,loan_id,gender,married,education,dependents


In [7]:
# Finding rows with the same loan_id value
customers[customers.duplicated(subset=['loan_id'], keep=False) == True]

,customer_id,loan_id,gender,married,education,dependents


As it can be seen, each row has unique **customer_id** and **loan_id**. We can also estimate the percentage of male and female customers in the dataset and get also the percentage of educated and married customers (in generall or grouped by gender).

In [8]:
customers.shape

(598, 6)

In [9]:
# Computing the percentages of male and female customers in the dataset
gender_customers = customers[["gender", "customer_id"]].groupby("gender").count()

perc_male_customers = 100.0*gender_customers.loc["Male", "customer_id"] / customers.shape[0]
perc_female_customers = 100.0*gender_customers.loc["Female", "customer_id"] / customers.shape[0]

print(f"The percentage of male customers is {perc_male_customers:.2f}% while female customers percentage is {perc_female_customers:.2f}%.")

The percentage of male customers is 81.44% while female customers percentage is 18.56%.


In [10]:
# Counting customers by grouping them into gender, married and education status
educated_married_customers = customers[["gender", "married", "education", "customer_id"]]. groupby(["gender", "married", "education"], as_index=False).count()
educated_married_customers

,gender,married,education,customer_id
0,Female,No,Graduate,66
1,Female,No,Not Graduate,14
2,Female,Yes,Graduate,25
3,Female,Yes,Not Graduate,6
4,Male,No,Graduate,99
5,Male,No,Not Graduate,31
6,Male,Yes,Graduate,275
7,Male,Yes,Not Graduate,82


In [11]:
financials.head()

,customer_id,total_income,credit_history
0,1,5849.0,1.0
1,2,6091.0,1.0
2,3,3000.0,1.0
3,4,4941.0,1.0
4,5,6000.0,1.0


In [12]:
financials.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   customer_id     598 non-null    int64  
 1   total_income    598 non-null    float64
 2   credit_history  549 non-null    float64
dtypes: float64(2), int64(1)
memory usage: 14.1 KB


In [13]:
loans.head()

,customer_id,loan_id,loan_amount,loan_amount_term,loan_status
0,1,LP001002,NaN,360.0,Y
1,2,LP001003,128.0,360.0,N
2,3,LP001005,66.0,360.0,Y
3,4,LP001006,120.0,360.0,Y
4,5,LP001008,141.0,360.0,Y


In [14]:
loans.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 598 entries, 0 to 597
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customer_id       598 non-null    int64  
 1   loan_id           598 non-null    object 
 2   loan_amount       577 non-null    float64
 3   loan_amount_term  584 non-null    float64
 4   loan_status       598 non-null    object 
dtypes: float64(2), int64(1), object(2)
memory usage: 23.5+ KB
